In [ ]:
pip install jmespath

# User Info

In [ ]:
import logging
logger = logging.getLogger(__name__)
logging.basicConfig(filename='example.log', encoding='utf-8', level=logging.DEBUG)

import json
import httpx

client = httpx.Client(
    headers={
        # this is internal ID of an instegram backend app. It doesn't change often.
        "x-ig-app-id": "936619743392459",
        # use browser-like features
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/62.0.3202.94 Safari/537.36",
        "Accept-Language": "en-US,en;q=0.9,ru;q=0.8",
        "Accept-Encoding": "gzip, deflate, br",
        "Accept": "*/*",
    }
)


def scrape_user(username: str):
    """Scrape Instagram user's data"""
    result = client.get(
        f"https://i.instagram.com/api/v1/users/web_profile_info/?username={username}",
    )
    data = json.loads(result.content)
    return data["data"]["user"]

# print(scrape_user("google"))

import jmespath
from typing import Dict
from urllib.parse import quote

def parse_user(data: Dict) -> Dict:
    """Parse instagram user's hidden web dataset for user's data"""
    logger.debug("parsing user data {}", data['username'])
    result = jmespath.search(
        """{
        name: full_name,
        username: username,
        id: id,
        category: category_name,
        business_category: business_category_name,
        phone: business_phone_number,
        email: business_email,
        bio: biography,
        bio_links: bio_links[].url,
        homepage: external_url,        
        followers: edge_followed_by.count,
        follows: edge_follow.count,
        facebook_id: fbid,
        is_private: is_private,
        is_verified: is_verified,
        profile_image: profile_pic_url_hd,
        video_count: edge_felix_video_timeline.count,
        videos: edge_felix_video_timeline.edges[].node.{
            id: id, 
            title: title,
            shortcode: shortcode,
            thumb: display_url,
            url: video_url,
            views: video_view_count,
            tagged: edge_media_to_tagged_user.edges[].node.user.username,
            captions: edge_media_to_caption.edges[].node.text,
            comments_count: edge_media_to_comment.count,
            comments_disabled: comments_disabled,
            taken_at: taken_at_timestamp,
            likes: edge_liked_by.count,
            location: location.name,
            duration: video_duration
        },
        image_count: edge_owner_to_timeline_media.count,
        images: edge_felix_video_timeline.edges[].node.{
            id: id, 
            title: title,
            shortcode: shortcode,
            src: display_url,
            url: video_url,
            views: video_view_count,
            tagged: edge_media_to_tagged_user.edges[].node.user.username,
            captions: edge_media_to_caption.edges[].node.text,
            comments_count: edge_media_to_comment.count,
            comments_disabled: comments_disabled,
            taken_at: taken_at_timestamp,
            likes: edge_liked_by.count,
            location: location.name,
            accesibility_caption: accessibility_caption,
            duration: video_duration
        },
        saved_count: edge_saved_media.count,
        collections_count: edge_saved_media.count,
        related_profiles: edge_related_profiles.edges[].node.username
    }""",
        data,
    )
    return result

In [98]:
# Test
username = 'sumer.noufouri' # 'vanelauck'
raw_user_info = scrape_user(username)
user_info = parse_user(raw_user_info)

--- Logging error ---
Traceback (most recent call last):
  File "c:\Users\juani\anaconda3\envs\time_series\lib\logging\__init__.py", line 1083, in emit
    msg = self.format(record)
  File "c:\Users\juani\anaconda3\envs\time_series\lib\logging\__init__.py", line 927, in format
    return fmt.format(record)
  File "c:\Users\juani\anaconda3\envs\time_series\lib\logging\__init__.py", line 663, in format
    record.message = record.getMessage()
  File "c:\Users\juani\anaconda3\envs\time_series\lib\logging\__init__.py", line 367, in getMessage
    msg = msg % self.args
TypeError: not all arguments converted during string formatting
Call stack:
  File "c:\Users\juani\anaconda3\envs\time_series\lib\runpy.py", line 197, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "c:\Users\juani\anaconda3\envs\time_series\lib\runpy.py", line 87, in _run_code
    exec(code, run_globals)
  File "c:\Users\juani\anaconda3\envs\time_series\lib\site-packages\ipykernel_launcher.py", l

In [101]:
user_info

{'name': 'Sumer Noufouri | Desarrollos Inmobiliarios',
 'username': 'sumer.noufouri',
 'id': '2031699985',
 'category': 'Real Estate Developer',
 'business_category': None,
 'phone': None,
 'email': None,
 'bio': '|  Te acompaño en la compra inteligente y segura de tu próximo departamento\n| + de 13 años de desarrollo\n| Proyectos 👇',
 'bio_links': ['http://www.go-building.com'],
 'homepage': 'http://www.go-building.com/',
 'followers': 14327,
 'follows': 1160,
 'facebook_id': '17841402039563962',
 'is_private': False,
 'is_verified': False,
 'profile_image': 'https://instagram.faep24-2.fna.fbcdn.net/v/t51.2885-19/438754664_1099698964583313_5735034941859737606_n.jpg?stp=dst-jpg_s320x320_tt6&_nc_ht=instagram.faep24-2.fna.fbcdn.net&_nc_cat=109&_nc_oc=Q6cZ2AEaChwa-nQFTAmyR4UzmU8ggw9bMljU0m240bABLbPkd7HpuiFw87Quh0YooPSNvLY&_nc_ohc=dY5YaBG0684Q7kNvgH0kHsT&_nc_gid=CdHnPVVAJawSDGtbW7U2Hw&edm=AOQ1c0wBAAAA&ccb=7-5&oh=00_AYF6bbeP28C7Xbe9rF2uRK_p4vqGkOJ3VRneGK-Uz5wyww&oe=67DF7191&_nc_sid=8b3546',

# Scrape Posts

## How to Scrape Instagram Posts?


In [32]:
INSTAGRAM_DOCUMENT_ID = "8845758582119845" # contst id for post documents instagram.com
shortcode = "CJ9KxZ2l8jT" # the post id

variables = {
    'shortcode':shortcode,'fetch_tagged_user_count':None,
    'hoisted_comment_id':None,'hoisted_reply_id':None
}
variables = quote(json.dumps(variables, separators=(',', ':')))
body = f"variables={variables}&doc_id={INSTAGRAM_DOCUMENT_ID}"

In [33]:
import httpx
import json
from typing import Dict
from urllib.parse import quote

INSTAGRAM_DOCUMENT_ID = "8845758582119845" # contst id for post documents instagram.com


def scrape_post(url_or_shortcode: str) -> Dict:
    """Scrape single Instagram post data"""
    if "http" in url_or_shortcode:
        shortcode = url_or_shortcode.split("/p/")[-1].split("/")[0]
    else:
        shortcode = url_or_shortcode
    print(f"scraping instagram post: {shortcode}")

    variables = quote(json.dumps({
        'shortcode':shortcode,'fetch_tagged_user_count':None,
        'hoisted_comment_id':None,'hoisted_reply_id':None
    }, separators=(',', ':')))
    body = f"variables={variables}&doc_id={INSTAGRAM_DOCUMENT_ID}"
    url = "https://www.instagram.com/graphql/query"

    result = httpx.post(
        url=url,
        headers={"content-type": "application/x-www-form-urlencoded"},
        data=body
    )
    data = json.loads(result.content)
    return data["data"]["xdt_shortcode_media"]

# Example usage:
posts = scrape_post("https://www.instagram.com/p/CuE2WNQs6vH/")

# save a JSON file
with open("result.json", "w",encoding="utf-8") as f:
    json.dump(posts, f, indent=2, ensure_ascii=False)

scraping instagram post: CuE2WNQs6vH


In [35]:
import jmespath
from typing import Dict

def parse_post(data: Dict) -> Dict:
    print("parsing post data {}", data['xdt_shortcode_media'])
    result = jmespath.search("""{
        id: id,
        shortcode: shortcode,
        dimensions: dimensions,
        src: display_url,
        src_attached: edge_sidecar_to_children.edges[].node.display_url,
        has_audio: has_audio,
        video_url: video_url,
        views: video_view_count,
        plays: video_play_count,
        likes: edge_media_preview_like.count,
        location: location.name,
        taken_at: taken_at_timestamp,
        related: edge_web_media_to_related_media.edges[].node.shortcode,
        type: product_type,
        video_duration: video_duration,
        music: clips_music_attribution_info,
        is_video: is_video,
        tagged_users: edge_media_to_tagged_user.edges[].node.user.username,
        captions: edge_media_to_caption.edges[].node.text,
        related_profiles: edge_related_profiles.edges[].node.username,
        comments_count: edge_media_to_parent_comment.count,
        comments_disabled: comments_disabled,
        comments_next_page: edge_media_to_parent_comment.page_info.end_cursor,
        comments: edge_media_to_parent_comment.edges[].node.{
            id: id,
            text: text,
            created_at: created_at,
            owner: owner.username,
            owner_verified: owner.is_verified,
            viewer_has_liked: viewer_has_liked,
            likes: edge_liked_by.count
        }
    }""", data)
    return result

In [41]:
results_parsed

NameError: name 'results_parsed' is not defined

## Scrape All Posts

'17841402039563962'

In [49]:
import json
import httpx
from urllib.parse import quote
from typing import Optional

INSTAGRAM_ACCOUNT_DOCUMENT_ID = "9310670392322965"

async def scrape_user_posts(username: str, page_size=12, max_pages: Optional[int] = None):
    """Scrape all posts of an Instagram user given the username."""
    base_url = "https://www.instagram.com/graphql/query"
    variables = {
        "after": None,
        "before": None,
        "data": {
            "count": page_size,
            "include_reel_media_seen_timestamp": True,
            "include_relationship_info": True,
            "latest_besties_reel_media": True,
            "latest_reel_media": True
        },
        "first": page_size,
        "last": None,
        "username": f"{username}",
        "__relay_internal__pv__PolarisIsLoggedInrelayprovider": True,
        "__relay_internal__pv__PolarisShareSheetV3relayprovider": True
    }

    prev_cursor = None
    _page_number = 1

    async with httpx.AsyncClient(timeout=httpx.Timeout(20.0)) as session:
        while True:
            body = f"variables={quote(json.dumps(variables, separators=(',', ':')))}&doc_id={INSTAGRAM_ACCOUNT_DOCUMENT_ID}"

            response = await session.post(
                base_url,
                data=body,
                headers={"content-type": "application/x-www-form-urlencoded"}
            )
            response.raise_for_status()
            data = response.json()

            with open("ts2.json", "w", encoding="utf-8") as f:
                json.dump(data, f, indent=2, ensure_ascii=False)

            posts = data["data"]["xdt_api__v1__feed__user_timeline_graphql_connection"]
            for post in posts["edges"]:
                yield post["node"]

            page_info = posts["page_info"]
            if not page_info["has_next_page"]:
                print(f"scraping posts page {_page_number}")
                break

            if page_info["end_cursor"] == prev_cursor:
                print("found no new posts, breaking")
                break

            prev_cursor = page_info["end_cursor"]
            variables["after"] = page_info["end_cursor"]
            _page_number += 1

            if max_pages and _page_number > max_pages:
                break



In [ ]:
c

'sumer.noufouri'

In [ ]:
# Example run:
if __name__ == "__main__":
    import asyncio

    async def main():
        posts = [post async for post in scrape_user_posts("sumer.noufouri", max_pages=3)]
        print(json.dumps(posts, indent=2, ensure_ascii=False))

    asyncio.run(main())

In [ ]:
posts = [post async for post in scrape_user_posts(username, max_pages=3)]
len(posts)

36

#### mentions

In [80]:
from collections import Counter

def scrape_hashtag_mentions(user_id, session: httpx.AsyncClient, page_limit:int=None):
    """find all hashtags user mentioned in their posts"""
    hashtags = Counter()
    hashtag_pattern = re.compile(r"#(\w+)")
    for post in scrape_user_posts(user_id, session=session, page_limit=page_limit):
        desc = '\n'.join(post['captions'])
        found = hashtag_pattern.findall(desc)
        for tag in found:
            hashtags[tag] += 1
    return hashtags

In [ ]:
# example
import json
import httpx

if __name__ == "__main__":
    with httpx.Client(timeout=httpx.Timeout(20.0)) as session:
        # if we only know the username but not user id we can scrape
        # the user profile to find the id:
        user_id = scrape_user("google")["id"]  # will result in: 1067259270
        # then we can scrape the hashtag profile
        hashtags = scrape_hastag_mentions(user_id, session, page_limit=5)
        # order results and print them as JSON:
        print(json.dumps(dict(h

In [95]:
session

In [91]:
%time
session  = httpx.Client(timeout=httpx.Timeout(20.0))
username ='vanelauck'
user_info = scrape_user(username)
user_id = user_info["id"]            
page_limit = 2

CPU times: total: 0 ns
Wall time: 0 ns


In [96]:
def scrape_hashtag_mentions(user_id, page_limit:int=None):
    """find all hashtags user mentioned in their posts"""
    hashtags = Counter()
    hashtag_pattern = re.compile(r"#(\w+)")
    session  = httpx.Client(timeout=httpx.Timeout(20.0))

    for post in scrape_user_posts(user_id, session=session, page_limit=page_limit):
        desc = '\n'.join(post['captions'])
        found = hashtag_pattern.findall(desc)
        for tag in found:
            hashtags[tag] += 1
    return hashtags

In [97]:

test = scrape_hashtag_mentions(user_id, page_limit)

TypeError: scrape_user_posts() got an unexpected keyword argument 'session'

In [72]:
posts[0]['caption']

{'has_translation': None,
 'created_at': 1720473203,
 'pk': '18113180815377767',
 'text': 'El consejo que me cambió la vida.\n\nSoy abogado. Un amigo me aconsejó con cariño y mucha visión: “Vos deberías ser desarrollador inmobiliario”. \n\nEntre las dos profesiones hay energías muy distintas.\n\nEl abogado recibe gente con problemas (generalmente). El desarrollador recibe gente con un sueño. \n\nY cuando podes ayudar a una persona a cumplir su sueño, ya no hay vuelta atrás, tu vida cambia.'}

In [73]:
# list(posts[0].keys())

In [ ]:
import  re
def 
hashtags = Counter()
hashtag_pattern = re.compile(r"#(\w+)")
for post in posts: # scrape_user_posts(user_id, session=session, page_limit=page_limit):
    desc = '\n'.join(post['caption'])
    found = hashtag_pattern.findall(desc)
    for tag in found:
        hashtags[tag] += 1
hashtags

Counter()

In [85]:
test = scrape_hashtag_mentions('vanelauck',  httpx.AsyncClient, page_limit=1)

TypeError: scrape_user_posts() got an unexpected keyword argument 'session'